# ChEMBL Scientific Data Lakehouse

This notebook demonstrates a Databricks and PySpark data engineering workflow using ChEMBL activity data.

The project covers:

- Reading scientific data stored in Parquet format
- Data inspection and profiling with PySpark
- Data quality validation
- Writing data to Delta Lake
- Creating and querying a Delta table
- Preparing the data for analytical transformations

The source data was previously ingested from the ChEMBL API and transformed into a structured Silver dataset using the Python and PySpark pipeline included in this repository.


## 1. Imports and Configuration

The following cell defines the libraries and paths used throughout the notebook.


In [0]:
from pyspark.sql import functions as F

PARQUET_PATH = "/Volumes/workspace/default/chembl_data/silver_parquet/silver_parquet/"
DELTA_PATH = "/Volumes/workspace/default/chembl_data/activity_delta"
TABLE_NAME = "workspace.default.chembl_activity"

## 2. Load Silver Data

The Silver dataset is stored as Parquet files in a Databricks Volume.

We load the data with Spark and use it as the input for the following profiling, quality checks, and Delta Lake operations.

In [0]:
df = spark.read.parquet(PARQUET_PATH)

print(f"Record count: {df.count()}")
df.printSchema()

display(df.limit(10))

## 3. Data Profiling

The following checks provide a basic overview of the dataset before writing it to Delta Lake.

We inspect the number of records, available columns, and the distribution of activity types.


In [0]:
print(f"Number of records:{df.count()}")
print(f"Number of columns:{len(df.columns)}")
display(df.groupBy("standard_type").count().orderBy(F.desc("count")))




## 4. Data Quality Checks

The following checks validate key data quality conditions before the dataset is written to Delta Lake.

We check for:
- missing activity identifiers
- missing activity types
- missing activity values
- duplicate activity identifiers

In [0]:
print(f"Number of missing activity identifiers:{df.filter(df.activity_id.isNull()).count()}")
print(f"Number of missing standard types:{df.filter(df.standard_type.isNull()).count()}")
print(f"Number of missing standard values:{df.filter(df.standard_value.isNull()).count()}")
print(f"Number of duplicated activity identifiers: {df.groupBy("activity_id").count().filter(F.col("count") > 1).count()}")


## 5. Save parquet files to delta format


In [0]:
df.write.format("delta").mode("overwrite").save(DELTA_PATH)


## 6. Read back delta format

In [0]:
delta_df = spark.read.format("delta").load(DELTA_PATH)
df.count()


## 7. Create delta table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME}
USING DELTA
AS
SELECT *
FROM delta.`{DELTA_PATH}`
""")

### Validate activity value status

Validate the transformation by checking the number of records with present and missing `standard_value`.

In [0]:
#df_transformed = df.withColumn('standard_value_scaled', F.col('standard_value')/1000)
#df_transformed.select('standard_value', 'standard_value_scaled').show()
df_transformed= df.withColumn('activity_value_status', F.when(F.col('standard_value').isNull(), 'missing').otherwise('present'))
df_transformed.groupBy("activity_value_status").count().show()

### Write transformed data to Delta

Persist the transformed DataFrame as a Delta table for downstream processing and analysis.

In [0]:
output_path = "workspace.default.chembl_activity_transformed"
df_transformed.write.format("delta").mode("overwrite").saveAsTable(output_path)

### Read transformed Delta table

Read the transformed dataset from the Delta table to verify that it was persisted correctly.

In [0]:
df_delta_transformed = spark.table(
    "workspace.default.chembl_activity_transformed"
)
df_delta_transformed.count()
df_delta_transformed.printSchema()
df_delta_transformed.limit(5).toPandas()

### Inspect Delta table history

Review the transaction history of the Delta table to examine recorded write operations and available table versions.

In [0]:
%sql
DESCRIBE HISTORY workspace.default.chembl_activity_transformed;

In [0]:
df_version_1 = spark.read.format("delta").option("versionAsOf",1).table("workspace.default.chembl_activity_transformed")
df_version_1.show(10)


Check activity_value_status statistics (compare with the results before saving the table )

In [0]:
df_delta_transformed.groupBy("activity_value_status").count().show()

**Validate Data Quality**
- Missing activity_id values.
- Duplicate activity_id records.
- Invalid or negative standard_value values.

In [0]:
print(f"Number of missing activity identifiers: {df_delta_transformed.filter(df_delta_transformed.activity_id.isNull()).count()}")
print(f"Number of duplicated activity identifiers: {df_delta_transformed.groupBy('activity_id').count().filter(F.col('count') > 1).count()}")
print(f"Number of invalid standard values: {df_delta_transformed.filter(
    (df_delta_transformed.standard_value.isNull()) | (df_delta_transformed.standard_value < 0))
    .count()}")


**Validate pChEMBL Values**

In [0]:
print(f"Number of missing pChEMBL Values: {df_delta_transformed.filter(df_delta_transformed.pchembl_value.isNull()).count()}")
print(f"Number of pChEMBL Values outside the range 0-14: {df_delta_transformed.filter((df_delta_transformed.pchembl_value > 14) | (df_delta_transformed.pchembl_value <0)).count()}")


 **IC50 Activity Filtering and Aggregation**

Filter IC50 activity records with non-null standard values, select relevant molecular and assay attributes, and aggregate activity data by assay and biological target.

The aggregation calculates the number of distinct molecules and the average standard value for each assay-target pair.

In [0]:
#df_delta_transformed.show()
#df_delta_transformed.groupBy('target_chembl_id').agg(F.count("molecule_chembl_id").alias("count_mol_id")).limit(50).show()
#display(df_delta_transformed.filter((df_delta_transformed.standard_type == 'IC50') & (df_delta_transformed.standard_value.isNotNull())))
columns =['activity_id', 'molecule_chembl_id', 'target_chembl_id', 'standard_value', 'assay_chembl_id']
df_filtered = df_delta_transformed.filter(
    (df_delta_transformed.standard_type == 'IC50') & (df_delta_transformed.standard_value.isNotNull()) & (df_delta_transformed.standard_units == "nM")
)
df_selected = df_filtered.select(*columns)
df_selected.groupBy('target_chembl_id').agg(
                    F.collect_list('molecule_chembl_id').alias('chembl_ids'), 
                    F.count('molecule_chembl_id').alias('count_mol_id')
                                            )
df_grouped = df_selected.groupBy('assay_chembl_id', 'target_chembl_id').agg(
                    F.collect_list('molecule_chembl_id').alias('chembl_ids'), 
                    F.countDistinct('molecule_chembl_id').alias('count_distinct_mol_id'),
                    F.avg('standard_value').alias('avg_standard_value')
).orderBy('assay_chembl_id', ascending=True).orderBy('avg_standard_value', ascending = True)                                         


**Save Gold Data to a delta table**

In [0]:
gold_output_path = "workspace.default.chembl_activity_gold"
df_grouped.write.format("delta").mode("overwrite").saveAsTable(gold_output_path)


**Read back and verify Gold Data**

In [0]:
df_gold = spark.table(gold_output_path)
df_gold.count()
df_gold.printSchema()
df_gold.show(10, truncate=False)

**Validate Gold Data**

In [0]:
distinct_pairs_count = df_gold.select("assay_chembl_id", "target_chembl_id").distinct().count()
print(f"Distinct pairs count: {distinct_pairs_count}")
all_records_count = df_gold.count()
print(f"counts match: {distinct_pairs_count == all_records_count}")
invalid_count_distinct_mol_id = df_gold.filter(F.col("count_distinct_mol_id").isNull() | (F.col("count_distinct_mol_id") < 0)).count()
print(f"number of invalid count_distinct_mol_id values (null or negative or zero): {invalid_count_distinct_mol_id}")
invalid_avg_standard_value = df_gold.filter(F.col("avg_standard_value").isNull() | (F.col("avg_standard_value") <= 0)).count()
print(f"number of invalid avg_standard_value values (null or negative): {invalid_avg_standard_value}")


